# Met Eyes Experiments

# Export Images

In [ ]:
import json
import numpy as np
import pandas as pd

from os import makedirs, path
from PIL import Image as PImage, ImageDraw as PImageDraw

from utils import mask_with_polygons, make_bins

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

IMG_FACES_DIR = f"{IMG_DIR}/faces"
IMG_EYES_DIR = f"{IMG_DIR}/eyes"
IMG_MASKED_DIR = f"{IMG_DIR}/masked"
IMG_PAIRS_DIR = f"{IMG_DIR}/pairs"

with open(f"{JSON_DIR}/faces.json", "r") as ifp:
  faces_data = json.load(ifp)["faces"]

with open(f"{JSON_DIR}/landmarks.json", "r") as ifp:
  landmarks_data = json.load(ifp)["landmarks"]

with open(f"{JSON_DIR}/mp_masks_definitions.json", "r") as ifp:
  mp_ldk_defs = json.load(ifp)

print("face images", len(faces_data))
print("landmark images", len(landmarks_data))

## Landmark Diagnostic

In [ ]:
sum_faces = 0
sum_landmarks = 0
fully_landmarked = 0
missed_ids = []

for obj in landmarks_data:
  if "faces" not in obj:
    print(f"{obj['objectID']} has no faces")
    continue
  if "yolo" not in obj["faces"]:
    print(f"{obj['objectID']} has no yolo")
    continue
  if "mp" not in obj["faces"]:
    print(f"{obj['objectID']} has no mp")
    continue

  sum_faces += obj["faces"]["yolo"]["count"]
  sum_landmarks += obj["faces"]["mp"]["count"]
  if obj["faces"]["yolo"]["count"] == obj["faces"]["mp"]["count"]:
    fully_landmarked += 1
  else:
    missed_ids.append(obj["objectID"])

print(f"total faces: {sum_faces}")
print(f"landmarked faces: {sum_landmarks}")

print(f"total images: {len(faces_data)}")
print(f"fully landmarked images: {fully_landmarked}")

### Size Stats

In [ ]:
EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

face_sizes = []
eye_sizes = []

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for lcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    face = obj["faces"]["yolo"]["xyxyn_sq"][lcnt]
    face_sizes.append((face[2] - face[0]) * iw)

    landmarks_np = np.array(landmarks) * img.size

    for eye_idxs in EYES:
      cx, cy = (landmarks_np[eye_idxs].mean(axis=0)).tolist()
      minx, miny = (landmarks_np[eye_idxs].min(axis=0)).tolist()
      maxx, maxy = (landmarks_np[eye_idxs].max(axis=0)).tolist()
      sx, sy = maxx - minx, maxy - miny
      eye_sizes.append(max(sx, sy))

face_bins = make_bins(face_sizes, bin_range=50)
eye_bins = make_bins(eye_sizes, bin_range=50)

face_size_cuts = pd.cut(pd.Series(face_sizes), bins=face_bins).value_counts().sort_index()
eye_size_cuts = pd.cut(pd.Series(eye_sizes), bins=eye_bins).value_counts().sort_index()

eye_size_qcuts = pd.qcut(pd.Series(eye_sizes), 8).value_counts().sort_index()

print(eye_size_cuts.to_string(), f"\ntotal: \t{eye_size_cuts.sum()}\n\n")
print(eye_size_qcuts.to_string(), f"\ntotal: \t{eye_size_cuts.sum()}")

### Mosaic: Missed (not landmarked) Faces

In [ ]:
NROWS, NCOLS = 11, 11
FACE_DIM = 128

missed_faces = PImage.new("RGB", (NCOLS * FACE_DIM, NROWS * FACE_DIM))
missed_face_cnt = 0

for oid in missed_ids:
  obj = [o for o in landmarks_data if o["objectID"] == oid][0]

  for lcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) > 0:
      continue

    face_img_cnt_str = f"000{lcnt}"[-3:]
    face_img_fname = f"{oid}_{face_img_cnt_str}"

    pimg = PImage.open(f"{IMG_FACES_DIR}/{face_img_fname}.jpg").resize((FACE_DIM, FACE_DIM))
    x = int(missed_face_cnt % NCOLS) * FACE_DIM
    y = int(missed_face_cnt / NCOLS) * FACE_DIM
    missed_faces.paste(pimg, (x, y))
    missed_face_cnt += 1

missed_faces.save("./imgs/missed_faces.jpg")
print(missed_face_cnt, "missed faces")
display(missed_faces)

### Mosaic: Export Landmarked Faces

In [ ]:
NROWS, NCOLS = 22, 22
FACE_DIM = 128
FACE_6 = mp_ldk_defs["FACE_6"]

marked_faces = PImage.new("RGB", (NCOLS * FACE_DIM, NROWS * FACE_DIM))
marked_face_cnt = 0

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for lcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    face = obj["faces"]["yolo"]["xyxyn_sq"][lcnt]
    fw, fh = (face[2] - face[0]), (face[3] - face[1])

    pimg = img.crop((face[0] * iw, face[1] * ih, face[2] * iw, face[3] * ih)).resize((FACE_DIM, FACE_DIM))
    draw = PImageDraw.Draw(pimg)
    for lidx in FACE_6:
      lmark = landmarks[lidx]
      cx = (lmark[0] - face[0]) / fw * FACE_DIM
      cy = (lmark[1] - face[1]) / fh * FACE_DIM
      draw.circle((cx, cy), radius=3, fill=(255,255,255), outline=(0,0,0), width=2)

    x = int(marked_face_cnt % NCOLS) * FACE_DIM
    y = int(marked_face_cnt / NCOLS) * FACE_DIM
    marked_faces.paste(pimg, (x, y))
    marked_face_cnt += 1

marked_faces.save("./imgs/marked_faces.jpg")
print(marked_face_cnt, "marked faces")
display(marked_faces)

### Mosaic: Export Eyes

In [ ]:
NROWS, NCOLS = 31, 31
EYE_DIM = 128
EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

marked_eyes = PImage.new("RGB", (NCOLS * EYE_DIM, NROWS * EYE_DIM))
marked_eye_cnt = 0

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for lcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    landmarks_np = np.array(landmarks) * img.size

    for eye_idxs in EYES:
      cx, cy = (landmarks_np[eye_idxs].mean(axis=0)).tolist()
      minx, miny = (landmarks_np[eye_idxs].min(axis=0)).tolist()
      maxx, maxy = (landmarks_np[eye_idxs].max(axis=0)).tolist()
      sx, sy = maxx - minx, maxy - miny
      dim_2 = max(sx, sy)//2
      eye_img = img.crop((cx - dim_2, cy - dim_2, cx + dim_2, cy + dim_2)).resize((EYE_DIM, EYE_DIM))

      x = int(marked_eye_cnt % NCOLS) * EYE_DIM
      y = int(marked_eye_cnt / NCOLS) * EYE_DIM
      marked_eyes.paste(eye_img, (x, y))
      marked_eye_cnt += 1

marked_eyes.save("./imgs/marked_eyes.jpg")
print(marked_eye_cnt, "marked eyes")
display(marked_eyes)

### Mosaic: Cropped Eyes

In [ ]:
NROWS, NCOLS = 31, 31
EYE_DIM = 128
EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

cropped_eyes = PImage.new("RGB", (NCOLS * EYE_DIM, NROWS * EYE_DIM))
cropped_eye_cnt = 0

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for lcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    landmarks_np = np.array(landmarks) * img.size
    eyes_points = [landmarks_np[eye_idxs] for eye_idxs in EYES]
    mimg = mask_with_polygons(img, eyes_points, 32)

    for eye_idxs in EYES:
      cx, cy = (landmarks_np[eye_idxs].mean(axis=0)).tolist()
      minx, miny = (landmarks_np[eye_idxs].min(axis=0)).tolist()
      maxx, maxy = (landmarks_np[eye_idxs].max(axis=0)).tolist()
      sx, sy = maxx - minx, maxy - miny
      dim_2 = int(1.25 * max(sx, sy)//2)
      eye_img = mimg.crop((cx - dim_2, cy - dim_2, cx + dim_2, cy + dim_2)).resize((EYE_DIM, EYE_DIM))

      x = int(cropped_eye_cnt % NCOLS) * EYE_DIM
      y = int(cropped_eye_cnt / NCOLS) * EYE_DIM
      cropped_eyes.paste(eye_img, (x, y))
      cropped_eye_cnt += 1

cropped_eyes.save("./imgs/cropped_eyes.jpg")
print(cropped_eye_cnt, "cropped eyes")
display(cropped_eyes)

### Export Cropped Eyes Images

In [ ]:
makedirs(IMG_EYES_DIR, exist_ok=True)

EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for fcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    face_cnt_str = f"000{fcnt}"[-3:]
    landmarks_np = np.array(landmarks) * img.size

    for ecnt,eye_idxs in enumerate(EYES):
      eye_cnt_str = "R" if ecnt == 0 else "L"
      eye_img_path = f"{IMG_EYES_DIR}/{oid}_{face_cnt_str}_{eye_cnt_str}.avif"
      if path.isfile(eye_img_path):
        continue

      eyes_points = [landmarks_np[eye_idxs]]
      mimg = mask_with_polygons(img, eyes_points, 64)
      cx, cy = (landmarks_np[eye_idxs].mean(axis=0)).tolist()
      minx, miny = (landmarks_np[eye_idxs].min(axis=0)).tolist()
      maxx, maxy = (landmarks_np[eye_idxs].max(axis=0)).tolist()
      sx, sy = maxx - minx, maxy - miny
      dim_2 = max(int(1.25 * max(sx, sy)//2), 1)
      eye_img = mimg.crop((cx - dim_2, cy - dim_2, cx + dim_2, cy + dim_2))
      eye_img.save(eye_img_path)

### Export Masked Eyes Images

In [ ]:
makedirs(IMG_MASKED_DIR, exist_ok=True)

EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  masked_img_path = f"{IMG_MASKED_DIR}/{oid}.avif"
  if path.isfile(masked_img_path):
    continue

  eyes_points = []
  for landmarks in obj["faces"]["mp"]["landmarks"]:
    if len(landmarks) < 1:
      continue

    landmarks_np = np.array(landmarks) * img.size
    eyes_points += [landmarks_np[eye_idxs].tolist() for eye_idxs in EYES]

    mimg = mask_with_polygons(img, eyes_points, 32)
    mimg.save(masked_img_path)

### Export Cropped Eye Pairs

In [ ]:
makedirs(IMG_PAIRS_DIR, exist_ok=True)

EYES = [mp_ldk_defs["A2b_R"], mp_ldk_defs["A2b_L"]]

for obj in landmarks_data:
  oid = obj["objectID"]
  img = PImage.open(f"{IMG_DIR}/original/{oid}.jpg")
  iw,ih = img.size

  for fcnt,landmarks in enumerate(obj["faces"]["mp"]["landmarks"]):
    if len(landmarks) < 1:
      continue

    face_cnt_str = f"000{fcnt}"[-3:]
    pair_img_path = f"{IMG_PAIRS_DIR}/{oid}_{face_cnt_str}.avif"
    if path.isfile(pair_img_path):
      continue

    landmarks_np = np.array(landmarks) * img.size
    pair_points = [landmarks_np[eye_idxs].tolist() for eye_idxs in EYES]

    pair_points_np = np.array(pair_points).reshape(-1, 2)
    minx, miny = (pair_points_np.min(axis=0)).tolist()
    maxx, maxy = (pair_points_np.max(axis=0)).tolist()
    cx, cy = (maxx + minx) / 2, (maxy + miny) / 2
    sx, sy = maxx - minx, maxy - miny
    w_2, h_2 = 1.25 * (maxx - minx) / 2, 1.25 * (maxy - miny) / 2

    scale = 2 if h_2 > 200 else (4 if h_2 > 100 else 8)
    mimg = mask_with_polygons(img, pair_points, detail=32, scale=scale)

    pair_img = mimg.crop((cx - w_2, cy - h_2, cx + w_2, cy + h_2))
    pair_img.save(pair_img_path)